In [1]:
# CELL 1: Day 3 - Setup and Environment
# This cell installs required packages and sets up the environment for Day 3
# Completely independent from previous days

import os
import sys
import warnings
import subprocess
import importlib.util
from datetime import datetime

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

print("DAY 3: MULTI-AGENT COLLABORATION")
print("=" * 60)
print(f"Started at: {datetime.now().isoformat()}")

required_packages = [
    ("crewai", "crewai"),
    ("langchain", "langchain"),
    ("langchain_core", "langchain-core"),
    ("langchain_community", "langchain-community"),
    ("wikipediaapi", "wikipedia-api"),
    ("requests", "requests"),
    ("pydantic", "pydantic"),
    ("typing_extensions", "typing_extensions"),
    ("gradio", "gradio"),
]

def install_package(package_name):
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        return True
    except subprocess.CalledProcessError:
        return False

def check_and_install_packages():
    installed = []
    failed = []
    
    for import_name, pip_name in required_packages:
        spec = importlib.util.find_spec(import_name)
        if spec is None:
            print(f"Installing: {pip_name}")
            if install_package(pip_name):
                installed.append(pip_name)
            else:
                failed.append(pip_name)
        else:
            print(f"Already installed: {pip_name}")
    
    if installed:
        print(f"\nSuccessfully installed: {', '.join(installed)}")
    if failed:
        print(f"\nFailed to install: {', '.join(failed)}")
    
    return len(failed) == 0

print("\nChecking and installing required packages...")
print("-" * 40)
installation_success = check_and_install_packages()
print("-" * 40)

if installation_success:
    print("All packages are ready.")
else:
    print("Some packages failed to install.")

print("\n" + "=" * 60)
print("Day 3 environment setup complete.")

DAY 3: MULTI-AGENT COLLABORATION
Started at: 2026-09-02T11:52:23.237133

Checking and installing required packages...
----------------------------------------
Installing: crewai
Already installed: langchain
Already installed: langchain-core
Installing: langchain-community
Installing: wikipedia-api
Already installed: requests
Already installed: pydantic
Already installed: typing_extensions
Already installed: gradio

----------------------------------------
All packages are ready.

Day 3 environment setup complete.


In [2]:
# CELL 2: Day 3 - Introduction to Multi-Agent Systems
# This cell provides an overview of what we will build today

print("DAY 3: MULTI-AGENT COLLABORATION")
print("=" * 60)
print()

print("TODAY'S OBJECTIVES:")
print("-" * 40)
print("1. Create Role-Based Agents (Research, Analysis, Writing)")
print("2. Implement Sequential Workflow (Agent 1 -> Agent 2 -> Agent 3)")
print("3. Implement Supervisor Workflow (Supervisor delegates tasks)")
print("4. Build Task Delegation System")
print("5. Generate Comprehensive Reports")
print()

print("WHAT WE WILL BUILD:")
print("-" * 40)
print("We will create a multi-agent system where:")
print("  - Research Agent: Searches and gathers information")
print("  - Analysis Agent: Analyzes and synthesizes information")
print("  - Writing Agent: Creates structured reports")
print("  - Supervisor Agent: Coordinates and delegates tasks")
print()

print("AGENT ROLES:")
print("-" * 40)
print("1. Research Agent:")
print("   - Goal: Find relevant information")
print("   - Tools: Wikipedia, Web Search")
print("   - Output: Raw information with sources")
print()
print("2. Analysis Agent:")
print("   - Goal: Analyze and synthesize information")
print("   - Tools: None (uses LLM/processing)")
print("   - Output: Structured analysis")
print()
print("3. Writing Agent:")
print("   - Goal: Create well-formatted reports")
print("   - Tools: None (uses LLM/formatting)")
print("   - Output: Professional report")
print()
print("4. Supervisor Agent:")
print("   - Goal: Coordinate all agents")
print("   - Tools: Task delegation")
print("   - Output: Final integrated result")
print()

print("WORKFLOW TYPES:")
print("-" * 40)
print("1. Sequential:")
print("   Research -> Analysis -> Writing")
print("   Each agent completes its task before next starts")
print()
print("2. Supervisor:")
print("   Supervisor assigns tasks to agents")
print("   Agents work in parallel or sequence")
print()
print("3. Hierarchical:")
print("   Supervisor -> Sub-supervisors -> Agents")
print("   Multiple levels of coordination")
print()

print("TOOLS WE WILL USE:")
print("-" * 40)
print("1. CrewAI - Framework for multi-agent collaboration")
print("2. LangChain - Tool integration and LLM calls")
print("3. Wikipedia API - Research and information gathering")
print("4. Calculator - Mathematical computations")
print("5. Web Search - Additional information retrieval")
print()

print("EXPECTED OUTPUTS:")
print("-" * 40)
print("1. 4 specialized agents with defined roles")
print("2. Sequential workflow execution")
print("3. Supervisor workflow execution")
print("4. Generated reports from multi-agent collaboration")
print("5. Performance comparison of different workflows")
print()

print("HOW THIS BUILDS ON PREVIOUS DAYS:")
print("-" * 40)
print("Day 1: Single agent with tools")
print("Day 2: Single agent with planning and memory")
print("Day 3: Multiple agents collaborating (Today)")
print("Day 4: Agent orchestration patterns")
print("Day 5: Complete production system")
print()

print("SAMPLE QUERIES WE WILL HANDLE TODAY:")
print("-" * 40)
print("1. 'Research artificial intelligence and create a report'")
print("   -> Sequential: Research -> Analyze -> Write")
print()
print("2. 'Find information about Python and machine learning'")
print("   -> Supervisor: Delegates to multiple research agents")
print()
print("3. 'Create a comprehensive report on quantum computing'")
print("   -> Full workflow: All agents collaborate")
print()

print("=" * 60)
print("Ready to start building Day 3!")
print("Run CELL 3 to create the base agent classes.")

DAY 3: MULTI-AGENT COLLABORATION

TODAY'S OBJECTIVES:
----------------------------------------
1. Create Role-Based Agents (Research, Analysis, Writing)
2. Implement Sequential Workflow (Agent 1 -> Agent 2 -> Agent 3)
3. Implement Supervisor Workflow (Supervisor delegates tasks)
4. Build Task Delegation System
5. Generate Comprehensive Reports

WHAT WE WILL BUILD:
----------------------------------------
We will create a multi-agent system where:
  - Research Agent: Searches and gathers information
  - Analysis Agent: Analyzes and synthesizes information
  - Writing Agent: Creates structured reports
  - Supervisor Agent: Coordinates and delegates tasks

AGENT ROLES:
----------------------------------------
1. Research Agent:
   - Goal: Find relevant information
   - Tools: Wikipedia, Web Search
   - Output: Raw information with sources

2. Analysis Agent:
   - Goal: Analyze and synthesize information
   - Tools: None (uses LLM/processing)
   - Output: Structured analysis

3. Writing Ag

In [3]:
# CELL 3: Base Agent Classes for Day 3
# This cell defines the base agent classes and tool implementations

import re
import math
import json
import time
import requests
import wikipediaapi
from typing import Dict, Any, Optional, List, Tuple
from datetime import datetime
from abc import ABC, abstractmethod

print("Building Base Agent Classes...")
print("=" * 60)

# ============================================================================
# TOOL IMPLEMENTATIONS
# ============================================================================

class CalculatorTool:
    """Calculator tool for mathematical operations."""
    name = "calculator"
    description = "Perform mathematical calculations"
    
    def _run(self, expression: str) -> str:
        try:
            expression = expression.strip()
            if not expression:
                return "Error: No expression provided"
            
            safe_context = {
                '__builtins__': {},
                'math': math,
                'sqrt': math.sqrt,
                'sin': math.sin,
                'cos': math.cos,
                'tan': math.tan,
                'log': math.log,
                'log10': math.log10,
                'abs': abs,
                'ceil': math.ceil,
                'floor': math.floor,
                'round': round,
                'pi': math.pi,
                'e': math.e
            }
            
            result = eval(expression, safe_context)
            return f"Result: {float(result)}"
        except Exception as e:
            return f"Error: {str(e)}"

class WikipediaTool:
    """Wikipedia search tool."""
    name = "wikipedia_search"
    description = "Search Wikipedia for information"
    
    def __init__(self):
        self._wiki = wikipediaapi.Wikipedia(
            language='en',
            user_agent='MultiAgent-Project/1.0'
        )
        self._cache = {}
    
    def _clean_text(self, text: str, max_length: int = 500) -> str:
        if not text:
            return "No content available."
        text = re.sub(r'\s+', ' ', text)
        if len(text) > max_length:
            text = text[:max_length] + "..."
        return text.strip()
    
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            page = self._wiki.page(query)
            if page.exists():
                return f"Wikipedia Article: {page.title}\n\n{self._clean_text(page.summary, 500)}\n\nURL: {page.fullurl}"
            
            results = list(self._wiki.search(query))[:max_results]
            if not results:
                return f"No Wikipedia results found for: {query}"
            
            output = f"Wikipedia Search Results for '{query}':\n\n"
            for idx, title in enumerate(results, 1):
                page = self._wiki.page(title)
                if page.exists():
                    output += f"{idx}. {title}\n"
                    output += f"   {self._clean_text(page.summary, 200)}\n"
                    output += f"   URL: {page.fullurl}\n\n"
            return output
        except Exception as e:
            return f"Error searching Wikipedia: {str(e)}"

class WebSearchTool:
    """Web search tool using Wikipedia API."""
    name = "web_search"
    description = "Search the web for information"
    
    def _run(self, query: str, max_results: int = 2) -> str:
        try:
            url = "https://en.wikipedia.org/w/api.php"
            params = {
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "srlimit": max_results,
                "srprop": "snippet"
            }
            headers = {"User-Agent": "MultiAgent-Project/1.0"}
            response = requests.get(url, params=params, headers=headers, timeout=10)
            
            if response.status_code != 200:
                return f"No results found for: {query}"
            
            data = response.json()
            results = data.get("query", {}).get("search", [])
            
            if not results:
                return f"No results found for: {query}"
            
            output = f"Search Results for '{query}':\n\n"
            for idx, item in enumerate(results, 1):
                title = item.get("title", "")
                snippet = re.sub(r'<[^>]+>', '', item.get("snippet", ""))
                output += f"{idx}. {title}\n"
                output += f"   {snippet[:200]}...\n\n"
            return output
        except Exception as e:
            return f"Error performing search: {str(e)}"

# ============================================================================
# TOOL REGISTRY
# ============================================================================

class ToolRegistry:
    """Registry for managing tools."""
    
    def __init__(self):
        self._tools = {}
    
    def register_tool(self, tool):
        self._tools[tool.name] = tool
        print(f"  Registered: {tool.name}")
    
    def get_tool(self, tool_name):
        return self._tools.get(tool_name)
    
    def list_tools(self):
        return list(self._tools.keys())
    
    def execute_tool(self, tool_name, **kwargs):
        tool = self.get_tool(tool_name)
        if not tool:
            return {"success": False, "error": f"Tool '{tool_name}' not found"}
        try:
            result = tool._run(**kwargs)
            return {"success": True, "result": result}
        except Exception as e:
            return {"success": False, "error": str(e)}

# ============================================================================
# BASE AGENT CLASS
# ============================================================================

class BaseAgent(ABC):
    """Abstract base class for all agents."""
    
    def __init__(self, name: str, role: str, goal: str, registry: ToolRegistry):
        self.name = name
        self.role = role
        self.goal = goal
        self.registry = registry
        self.memory = []
        self.output = None
        
    @abstractmethod
    def execute(self, input_data: Any) -> Any:
        """Execute the agent's primary function."""
        pass
    
    def get_tool(self, tool_name: str):
        """Get a tool from the registry."""
        return self.registry.get_tool(tool_name)
    
    def execute_tool(self, tool_name: str, **kwargs):
        """Execute a tool."""
        return self.registry.execute_tool(tool_name, **kwargs)
    
    def store_memory(self, content: str):
        """Store information in agent memory."""
        self.memory.append({
            "timestamp": datetime.now().isoformat(),
            "content": content
        })
    
    def get_memory(self) -> List[Dict]:
        """Get agent memory."""
        return self.memory
    
    def __str__(self):
        return f"{self.name} ({self.role}): {self.goal}"

# ============================================================================
# BUILD AND TEST
# ============================================================================

print("\nCreating tool registry...")
registry = ToolRegistry()

print("\nRegistering tools:")
registry.register_tool(WikipediaTool())
registry.register_tool(WebSearchTool())
registry.register_tool(CalculatorTool())

print(f"\nTools available: {registry.list_tools()}")

print("\n" + "=" * 60)
print("Base agent classes created successfully.")

Building Base Agent Classes...

Creating tool registry...

Registering tools:
  Registered: wikipedia_search
  Registered: web_search
  Registered: calculator

Tools available: ['wikipedia_search', 'web_search', 'calculator']

Base agent classes created successfully.


In [4]:
# CELL 4: Specialized Agents - Research, Analysis, Writing
# This cell creates specialized agents with specific roles

import json
from typing import Dict, Any, Optional, List
from datetime import datetime

print("Creating Specialized Agents...")
print("=" * 60)

# ============================================================================
# RESEARCH AGENT
# ============================================================================

class ResearchAgent(BaseAgent):
    """
    Research Agent: Searches and gathers information from various sources.
    """
    
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="ResearchAgent",
            role="Information Gatherer",
            goal="Find and gather relevant information from multiple sources",
            registry=registry
        )
        self.sources_used = []
    
    def execute(self, topic: str) -> Dict[str, Any]:
        """
        Execute research on a given topic.
        
        Args:
            topic: The topic to research
            
        Returns:
            Dictionary containing research results
        """
        print(f"\n[{self.name}] Researching: {topic}")
        print("-" * 40)
        
        results = {
            "topic": topic,
            "sources": [],
            "information": [],
            "timestamp": datetime.now().isoformat()
        }
        
        # Try Wikipedia first
        print("  Searching Wikipedia...")
        wiki_result = self.execute_tool("wikipedia_search", query=topic)
        if wiki_result.get('success') and wiki_result.get('result'):
            results["information"].append({
                "source": "Wikipedia",
                "content": wiki_result['result'],
                "timestamp": datetime.now().isoformat()
            })
            results["sources"].append("Wikipedia")
            self.sources_used.append("Wikipedia")
            print("  Found Wikipedia information")
        
        # Try Web Search
        print("  Searching Web...")
        web_result = self.execute_tool("web_search", query=topic)
        if web_result.get('success') and web_result.get('result'):
            results["information"].append({
                "source": "Web Search",
                "content": web_result['result'],
                "timestamp": datetime.now().isoformat()
            })
            results["sources"].append("Web Search")
            self.sources_used.append("Web Search")
            print("  Found Web information")
        
        # If no results, provide a summary
        if not results["information"]:
            results["information"].append({
                "source": "System",
                "content": f"No specific information found for '{topic}'. Please try a more specific query.",
                "timestamp": datetime.now().isoformat()
            })
        
        # Store in memory
        self.store_memory(f"Researched: {topic}")
        self.store_memory(f"Sources used: {', '.join(results['sources'])}")
        
        results["summary"] = self._generate_summary(results["information"])
        self.output = results
        
        print(f"  Research complete. Found {len(results['information'])} sources.")
        return results
    
    def _generate_summary(self, information: List[Dict]) -> str:
        """Generate a summary of the research findings."""
        if not information:
            return "No information available."
        
        summary = f"Research found {len(information)} sources:\n"
        for idx, info in enumerate(information, 1):
            source = info.get('source', 'Unknown')
            content = info.get('content', '')
            summary += f"{idx}. {source}: {content[:150]}...\n"
        
        return summary

# ============================================================================
# ANALYSIS AGENT
# ============================================================================

class AnalysisAgent(BaseAgent):
    """
    Analysis Agent: Analyzes and synthesizes information from research.
    """
    
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="AnalysisAgent",
            role="Information Analyst",
            goal="Analyze and synthesize information into structured insights",
            registry=registry
        )
    
    def execute(self, research_results: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute analysis on research results.
        
        Args:
            research_results: Results from the Research Agent
            
        Returns:
            Dictionary containing analysis results
        """
        print(f"\n[{self.name}] Analyzing research data...")
        print("-" * 40)
        
        topic = research_results.get('topic', 'Unknown')
        information = research_results.get('information', [])
        
        analysis = {
            "topic": topic,
            "key_findings": [],
            "insights": [],
            "recommendations": [],
            "timestamp": datetime.now().isoformat()
        }
        
        # Process each piece of information
        for info in information:
            content = info.get('content', '')
            
            # Extract key points (simplified)
            key_points = self._extract_key_points(content)
            if key_points:
                analysis["key_findings"].extend(key_points)
            
            # Generate insights
            insights = self._generate_insights(content, topic)
            if insights:
                analysis["insights"].extend(insights)
        
        # Generate recommendations
        analysis["recommendations"] = self._generate_recommendations(analysis)
        
        # Store in memory
        self.store_memory(f"Analyzed: {topic}")
        self.store_memory(f"Key findings: {len(analysis['key_findings'])}")
        self.store_memory(f"Insights: {len(analysis['insights'])}")
        
        self.output = analysis
        
        print(f"  Analysis complete. Found {len(analysis['key_findings'])} key findings.")
        print(f"  Generated {len(analysis['insights'])} insights.")
        return analysis
    
    def _extract_key_points(self, content: str) -> List[str]:
        """Extract key points from content."""
        points = []
        
        # Look for bullet points or numbered lists
        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            if line and (line.startswith('•') or line.startswith('-') or 
                         line.startswith('1.') or line.startswith('2.') or
                         line.startswith('3.') or line.startswith('4.') or
                         line.startswith('5.')):
                points.append(line)
        
        # If no bullet points found, create simple ones
        if not points and len(content) > 50:
            sentences = content.split('.')
            for sentence in sentences[:3]:
                sentence = sentence.strip()
                if len(sentence) > 20:
                    points.append(sentence[:100] + "...")
        
        return points[:5]  # Limit to 5 key points
    
    def _generate_insights(self, content: str, topic: str) -> List[str]:
        """Generate insights from content."""
        insights = []
        
        # Look for patterns and connections
        if "AI" in content or "artificial intelligence" in content.lower():
            insights.append(f"{topic} is related to artificial intelligence")
        
        if "machine learning" in content.lower() or "ML" in content:
            insights.append(f"Machine learning is a key component of {topic}")
        
        if "data" in content.lower():
            insights.append(f"Data plays a crucial role in {topic}")
        
        # Add general insight
        if len(content) > 100 and not insights:
            insights.append(f"{topic} is a significant field with multiple applications")
        
        return insights
    
    def _generate_recommendations(self, analysis: Dict) -> List[str]:
        """Generate recommendations based on analysis."""
        recommendations = []
        
        if analysis["key_findings"]:
            recommendations.append("Focus on the key findings identified")
        
        if analysis["insights"]:
            recommendations.append("Consider the insights for deeper understanding")
        
        if len(analysis["key_findings"]) < 2:
            recommendations.append("Conduct additional research on this topic")
        
        recommendations.append("Document findings for future reference")
        
        return recommendations

# ============================================================================
# WRITING AGENT
# ============================================================================

class WritingAgent(BaseAgent):
    """
    Writing Agent: Creates well-formatted reports and documentation.
    """
    
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="WritingAgent",
            role="Report Writer",
            goal="Create well-structured, professional reports and documentation",
            registry=registry
        )
    
    def execute(self, analysis_results: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute report writing based on analysis results.
        
        Args:
            analysis_results: Results from the Analysis Agent
            
        Returns:
            Dictionary containing the final report
        """
        print(f"\n[{self.name}] Writing report...")
        print("-" * 40)
        
        topic = analysis_results.get('topic', 'Unknown')
        findings = analysis_results.get('key_findings', [])
        insights = analysis_results.get('insights', [])
        recommendations = analysis_results.get('recommendations', [])
        
        report = {
            "topic": topic,
            "title": self._generate_title(topic),
            "executive_summary": self._generate_executive_summary(topic, findings),
            "findings": findings,
            "insights": insights,
            "recommendations": recommendations,
            "timestamp": datetime.now().isoformat()
        }
        
        # Generate full report
        report["full_report"] = self._generate_full_report(report)
        
        # Store in memory
        self.store_memory(f"Wrote report on: {topic}")
        self.store_memory(f"Report length: {len(report['full_report'])} characters")
        
        self.output = report
        
        print(f"  Report complete. Length: {len(report['full_report'])} characters.")
        return report
    
    def _generate_title(self, topic: str) -> str:
        """Generate a title for the report."""
        titles = [
            f"Comprehensive Report on {topic}",
            f"Analysis and Insights: {topic}",
            f"Research Findings: {topic}",
            f"Understanding {topic}: A Detailed Report"
        ]
        return titles[hash(topic) % len(titles)]
    
    def _generate_executive_summary(self, topic: str, findings: List[str]) -> str:
        """Generate an executive summary."""
        if not findings:
            return f"No significant findings discovered for {topic}."
        
        summary = f"This report provides an analysis of {topic}. "
        if len(findings) >= 3:
            summary += f"Key findings include: {', '.join(findings[:3])}. "
        else:
            summary += f"Key findings include: {'; '.join(findings)}. "
        summary += "This analysis is based on information from multiple sources."
        
        return summary
    
    def _generate_full_report(self, report: Dict) -> str:
        """Generate the full report in a structured format."""
        lines = []
        
        # Title
        lines.append("=" * 60)
        lines.append(report['title'])
        lines.append("=" * 60)
        lines.append("")
        
        # Timestamp
        lines.append(f"Generated: {report['timestamp']}")
        lines.append("")
        
        # Executive Summary
        lines.append("EXECUTIVE SUMMARY")
        lines.append("-" * 40)
        lines.append(report['executive_summary'])
        lines.append("")
        
        # Key Findings
        lines.append("KEY FINDINGS")
        lines.append("-" * 40)
        if report['findings']:
            for i, finding in enumerate(report['findings'], 1):
                lines.append(f"{i}. {finding}")
        else:
            lines.append("No specific findings were identified.")
        lines.append("")
        
        # Insights
        lines.append("INSIGHTS")
        lines.append("-" * 40)
        if report['insights']:
            for insight in report['insights']:
                lines.append(f"• {insight}")
        else:
            lines.append("No specific insights were generated.")
        lines.append("")
        
        # Recommendations
        lines.append("RECOMMENDATIONS")
        lines.append("-" * 40)
        if report['recommendations']:
            for i, rec in enumerate(report['recommendations'], 1):
                lines.append(f"{i}. {rec}")
        else:
            lines.append("No recommendations available.")
        lines.append("")
        
        lines.append("=" * 60)
        lines.append("END OF REPORT")
        lines.append("=" * 60)
        
        return "\n".join(lines)

# ============================================================================
# BUILD AND TEST
# ============================================================================

print("\nCreating specialized agents...")
research_agent = ResearchAgent(registry)
analysis_agent = AnalysisAgent(registry)
writing_agent = WritingAgent(registry)

print(f"\nAgents created:")
print(f"  - {research_agent}")
print(f"  - {analysis_agent}")
print(f"  - {writing_agent}")

print("\n" + "=" * 60)
print("Specialized agents created successfully.")

Creating Specialized Agents...

Creating specialized agents...

Agents created:
  - ResearchAgent (Information Gatherer): Find and gather relevant information from multiple sources
  - AnalysisAgent (Information Analyst): Analyze and synthesize information into structured insights
  - WritingAgent (Report Writer): Create well-structured, professional reports and documentation

Specialized agents created successfully.


In [5]:
# CELL 5: Supervisor Agent and Workflow Management (Fixed)
# This cell creates the Supervisor Agent and manages workflows

from typing import Dict, Any, Optional, List, Callable
from datetime import datetime
import time

print("Creating Supervisor Agent and Workflows...")
print("=" * 60)

# ============================================================================
# SUPERVISOR AGENT
# ============================================================================

class SupervisorAgent(BaseAgent):
    """
    Supervisor Agent: Coordinates and delegates tasks to other agents.
    """
    
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="SupervisorAgent",
            role="Orchestrator",
            goal="Coordinate and delegate tasks to specialized agents for optimal results",
            registry=registry
        )
        self.agents = {}
        self.workflow_history = []
    
    def execute(self, input_data: Any) -> Dict[str, Any]:
        """
        Execute the supervisor agent's primary function.
        This is required by the BaseAgent abstract class.
        
        Args:
            input_data: The input data (query or task)
            
        Returns:
            Result from the supervisor workflow
        """
        if isinstance(input_data, str):
            return self.execute_supervisor_workflow(input_data)
        else:
            return self.execute_supervisor_workflow(str(input_data))
    
    def register_agent(self, agent: BaseAgent) -> None:
        """
        Register an agent with the supervisor.
        
        Args:
            agent: The agent to register
        """
        self.agents[agent.name] = agent
        print(f"  Registered: {agent.name}")
    
    def register_agents(self, agents: List[BaseAgent]) -> None:
        """
        Register multiple agents.
        
        Args:
            agents: List of agents to register
        """
        for agent in agents:
            self.register_agent(agent)
    
    def get_agent(self, agent_name: str) -> Optional[BaseAgent]:
        """Get a registered agent by name."""
        return self.agents.get(agent_name)
    
    def list_agents(self) -> List[str]:
        """List all registered agents."""
        return list(self.agents.keys())
    
    def delegate_task(self, agent_name: str, input_data: Any) -> Dict[str, Any]:
        """
        Delegate a task to a specific agent.
        
        Args:
            agent_name: Name of the agent to delegate to
            input_data: Data to pass to the agent
            
        Returns:
            Result from the agent
        """
        agent = self.get_agent(agent_name)
        if not agent:
            return {"error": f"Agent '{agent_name}' not found"}
        
        print(f"\n[Supervisor] Delegating to: {agent_name}")
        start_time = time.time()
        
        try:
            result = agent.execute(input_data)
            execution_time = time.time() - start_time
            
            # Record in history
            self.workflow_history.append({
                "agent": agent_name,
                "input": input_data,
                "output": result,
                "execution_time": execution_time,
                "timestamp": datetime.now().isoformat()
            })
            
            return result
        except Exception as e:
            error_result = {"error": str(e)}
            self.workflow_history.append({
                "agent": agent_name,
                "input": input_data,
                "output": error_result,
                "error": str(e),
                "timestamp": datetime.now().isoformat()
            })
            return error_result
    
    def execute_sequential(self, query: str) -> Dict[str, Any]:
        """
        Execute a sequential workflow: Research -> Analysis -> Writing.
        
        Args:
            query: The user query
            
        Returns:
            Final result after sequential execution
        """
        print("\n" + "=" * 60)
        print(f"SEQUENTIAL WORKFLOW")
        print(f"Query: {query}")
        print("=" * 60)
        
        workflow_result = {
            "query": query,
            "workflow_type": "sequential",
            "steps": [],
            "final_result": None,
            "timestamp": datetime.now().isoformat()
        }
        
        # Step 1: Research
        print("\n[Step 1] Research Agent")
        print("-" * 40)
        research_result = self.delegate_task("ResearchAgent", query)
        workflow_result["steps"].append({
            "step": 1,
            "agent": "ResearchAgent",
            "result": research_result
        })
        
        if "error" in research_result:
            workflow_result["final_result"] = {"error": research_result["error"]}
            return workflow_result
        
        # Step 2: Analysis
        print("\n[Step 2] Analysis Agent")
        print("-" * 40)
        analysis_result = self.delegate_task("AnalysisAgent", research_result)
        workflow_result["steps"].append({
            "step": 2,
            "agent": "AnalysisAgent",
            "result": analysis_result
        })
        
        if "error" in analysis_result:
            workflow_result["final_result"] = {"error": analysis_result["error"]}
            return workflow_result
        
        # Step 3: Writing
        print("\n[Step 3] Writing Agent")
        print("-" * 40)
        writing_result = self.delegate_task("WritingAgent", analysis_result)
        workflow_result["steps"].append({
            "step": 3,
            "agent": "WritingAgent",
            "result": writing_result
        })
        
        workflow_result["final_result"] = writing_result
        
        # Store in supervisor memory
        self.store_memory(f"Sequential workflow completed for: {query}")
        
        print("\n" + "=" * 60)
        print("SEQUENTIAL WORKFLOW COMPLETE")
        print("=" * 60)
        
        return workflow_result
    
    def execute_supervisor_workflow(self, query: str) -> Dict[str, Any]:
        """
        Execute a supervisor workflow where supervisor delegates tasks.
        
        Args:
            query: The user query
            
        Returns:
            Final result
        """
        print("\n" + "=" * 60)
        print(f"SUPERVISOR WORKFLOW")
        print(f"Query: {query}")
        print("=" * 60)
        
        workflow_result = {
            "query": query,
            "workflow_type": "supervisor",
            "tasks": [],
            "final_result": None,
            "timestamp": datetime.now().isoformat()
        }
        
        # Analyze the query to determine which agents to use
        tasks = self._analyze_query_for_tasks(query)
        
        print(f"\n[Supervisor] Planning tasks:")
        for task in tasks:
            print(f"  - {task['agent']}: {task['description']}")
        
        # Execute each task
        results = []
        for task in tasks:
            print(f"\n[Supervisor] Executing task: {task['description']}")
            result = self.delegate_task(task['agent'], task['input'])
            results.append({
                "task": task['description'],
                "agent": task['agent'],
                "result": result
            })
            workflow_result["tasks"].append({
                "agent": task['agent'],
                "input": task['input'],
                "result": result
            })
        
        # Combine results
        final_result = self._combine_results(results, query)
        workflow_result["final_result"] = final_result
        
        self.store_memory(f"Supervisor workflow completed for: {query}")
        
        print("\n" + "=" * 60)
        print("SUPERVISOR WORKFLOW COMPLETE")
        print("=" * 60)
        
        return workflow_result
    
    def _analyze_query_for_tasks(self, query: str) -> List[Dict[str, Any]]:
        """
        Analyze the query to determine which agents should be used.
        
        Args:
            query: The user query
            
        Returns:
            List of tasks with agent assignments
        """
        query_lower = query.lower()
        tasks = []
        
        # Always start with research
        tasks.append({
            "agent": "ResearchAgent",
            "input": query,
            "description": "Research the topic"
        })
        
        # If query seems complex, add analysis
        if len(query.split()) > 5 or "analyze" in query_lower or "explain" in query_lower:
            tasks.append({
                "agent": "AnalysisAgent",
                "input": None,
                "description": "Analyze research findings"
            })
        
        # If query asks for report or summary
        if "report" in query_lower or "summary" in query_lower or "write" in query_lower:
            tasks.append({
                "agent": "WritingAgent",
                "input": None,
                "description": "Generate report"
            })
        
        # If no specific tasks, default to research and summary
        if len(tasks) == 1:
            tasks.append({
                "agent": "WritingAgent",
                "input": None,
                "description": "Summarize findings"
            })
        
        return tasks
    
    def _combine_results(self, results: List[Dict], query: str) -> Dict[str, Any]:
        """
        Combine results from multiple agents.
        
        Args:
            results: List of agent results
            query: Original query
            
        Returns:
            Combined result
        """
        combined = {
            "query": query,
            "sources": [],
            "findings": [],
            "report": "",
            "summary": ""
        }
        
        for result in results:
            agent = result.get('agent', 'Unknown')
            data = result.get('result', {})
            
            if isinstance(data, dict):
                if 'sources' in data:
                    combined['sources'].extend(data.get('sources', []))
                if 'key_findings' in data:
                    combined['findings'].extend(data.get('key_findings', []))
                if 'full_report' in data:
                    combined['report'] = data.get('full_report', '')
                if 'summary' in data:
                    combined['summary'] = data.get('summary', '')
        
        # Generate final report if not present
        if not combined['report'] and combined['findings']:
            combined['report'] = self._generate_final_report(combined, query)
        
        return combined
    
    def _generate_final_report(self, combined: Dict, query: str) -> str:
        """Generate a final report from combined results."""
        lines = []
        lines.append("=" * 60)
        lines.append(f"FINAL REPORT: {query}")
        lines.append("=" * 60)
        lines.append("")
        lines.append(f"Generated: {datetime.now().isoformat()}")
        lines.append("")
        
        if combined.get('sources'):
            lines.append("SOURCES CONSULTED:")
            lines.append("-" * 40)
            for source in combined['sources']:
                lines.append(f"  - {source}")
            lines.append("")
        
        if combined.get('findings'):
            lines.append("KEY FINDINGS:")
            lines.append("-" * 40)
            for i, finding in enumerate(combined['findings'], 1):
                lines.append(f"{i}. {finding}")
            lines.append("")
        
        if combined.get('summary'):
            lines.append("SUMMARY:")
            lines.append("-" * 40)
            lines.append(combined['summary'])
            lines.append("")
        
        lines.append("=" * 60)
        return "\n".join(lines)

    def get_workflow_history(self) -> List[Dict]:
        """Get workflow history."""
        return self.workflow_history

# ============================================================================
# BUILD AND TEST
# ============================================================================

print("\nCreating Supervisor Agent...")
supervisor = SupervisorAgent(registry)

print("\nRegistering agents with supervisor:")
supervisor.register_agents([research_agent, analysis_agent, writing_agent])

print(f"\nRegistered agents: {supervisor.list_agents()}")

# Test sequential workflow
print("\n" + "=" * 60)
print("TESTING SEQUENTIAL WORKFLOW")
print("=" * 60)

test_query = "artificial intelligence"
result = supervisor.execute_sequential(test_query)

print("\nFinal Report Preview:")
if result['final_result'] and isinstance(result['final_result'], dict):
    print(result['final_result'].get('full_report', 'No report generated')[:500] + "...")
else:
    print(str(result['final_result'])[:500] + "...")

print("\n" + "=" * 60)
print("Supervisor Agent and workflows created successfully.")

Creating Supervisor Agent and Workflows...

Creating Supervisor Agent...

Registering agents with supervisor:
  Registered: ResearchAgent
  Registered: AnalysisAgent
  Registered: WritingAgent

Registered agents: ['ResearchAgent', 'AnalysisAgent', 'WritingAgent']

TESTING SEQUENTIAL WORKFLOW

SEQUENTIAL WORKFLOW
Query: artificial intelligence

[Step 1] Research Agent
----------------------------------------

[Supervisor] Delegating to: ResearchAgent

[ResearchAgent] Researching: artificial intelligence
----------------------------------------
  Searching Wikipedia...
  Found Wikipedia information
  Searching Web...
  Found Web information
  Research complete. Found 2 sources.

[Step 2] Analysis Agent
----------------------------------------

[Supervisor] Delegating to: AnalysisAgent

[AnalysisAgent] Analyzing research data...
----------------------------------------
  Analysis complete. Found 4 key findings.
  Generated 2 insights.

[Step 3] Writing Agent
------------------------------

In [6]:
# CELL 6: Gradio Interface for Multi-Agent System
# This cell creates an interactive UI for the multi-agent system

import gradio as gr
from datetime import datetime
import json

print("Building Multi-Agent System Interface...")
print("=" * 60)

class MultiAgentInterface:
    """
    Interface for the multi-agent system.
    """
    
    def __init__(self, supervisor_agent):
        self.supervisor = supervisor_agent
        self.conversation_history = []
        self.workflow_results = []
    
    def process_query(self, query: str, workflow_type: str = "sequential") -> str:
        """
        Process a query using the selected workflow type.
        
        Args:
            query: User query
            workflow_type: Type of workflow to use
            
        Returns:
            Formatted response
        """
        if not query or query.strip() == "":
            return "Please enter a query."
        
        print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Query: {query}")
        print(f"Workflow Type: {workflow_type}")
        
        try:
            # Execute workflow
            if workflow_type == "sequential":
                result = self.supervisor.execute_sequential(query)
            else:
                result = self.supervisor.execute_supervisor_workflow(query)
            
            # Store in history
            self.workflow_results.append({
                "query": query,
                "workflow_type": workflow_type,
                "result": result,
                "timestamp": datetime.now().isoformat()
            })
            
            # Format response
            response = self._format_response(result, query)
            return response
            
        except Exception as e:
            error_msg = f"Error processing query: {str(e)}"
            print(error_msg)
            return error_msg
    
    def _format_response(self, result: Dict, query: str) -> str:
        """Format the workflow result as a readable response."""
        lines = []
        
        lines.append("=" * 60)
        lines.append(f"QUERY: {query}")
        lines.append("=" * 60)
        lines.append("")
        
        # Get final result
        final_result = result.get('final_result', {})
        
        if isinstance(final_result, dict):
            # Check for report
            if 'full_report' in final_result:
                lines.append(final_result['full_report'])
            elif 'report' in final_result:
                lines.append(final_result['report'])
            else:
                # Display findings
                lines.append("FINDINGS:")
                lines.append("-" * 40)
                if 'findings' in final_result:
                    for i, finding in enumerate(final_result['findings'], 1):
                        lines.append(f"{i}. {finding}")
                elif 'key_findings' in final_result:
                    for i, finding in enumerate(final_result['key_findings'], 1):
                        lines.append(f"{i}. {finding}")
                else:
                    lines.append(str(final_result))
        else:
            lines.append(str(final_result))
        
        lines.append("")
        lines.append("-" * 60)
        
        # Add workflow summary
        lines.append("WORKFLOW SUMMARY:")
        lines.append(f"  Type: {result.get('workflow_type', 'Unknown')}")
        lines.append(f"  Steps: {len(result.get('steps', result.get('tasks', [])))}")
        
        if 'steps' in result:
            for step in result['steps']:
                lines.append(f"    - {step.get('agent', 'Unknown')}")
        elif 'tasks' in result:
            for task in result['tasks']:
                lines.append(f"    - {task.get('agent', 'Unknown')}")
        
        lines.append("=" * 60)
        
        return "\n".join(lines)
    
    def get_agent_info(self) -> str:
        """Get information about all agents."""
        agents = self.supervisor.list_agents()
        
        lines = []
        lines.append("=" * 60)
        lines.append("MULTI-AGENT SYSTEM INFORMATION")
        lines.append("=" * 60)
        lines.append("")
        
        lines.append("REGISTERED AGENTS:")
        lines.append("-" * 40)
        for agent_name in agents:
            agent = self.supervisor.get_agent(agent_name)
            if agent:
                lines.append(f"  Name: {agent.name}")
                lines.append(f"  Role: {agent.role}")
                lines.append(f"  Goal: {agent.goal}")
                lines.append("")
        
        lines.append("WORKFLOW TYPES:")
        lines.append("-" * 40)
        lines.append("  Sequential: Research -> Analysis -> Writing")
        lines.append("  Supervisor: Supervisor delegates tasks to agents")
        lines.append("")
        
        lines.append("WORKFLOW HISTORY:")
        lines.append("-" * 40)
        history = self.supervisor.get_workflow_history()
        if history:
            lines.append(f"  Total workflows: {len(history)}")
            for entry in history[-3:]:
                lines.append(f"    - {entry.get('agent', 'Unknown')}: {entry.get('timestamp', '')[:19]}")
        else:
            lines.append("  No workflows executed yet.")
        
        lines.append("=" * 60)
        return "\n".join(lines)
    
    def clear_history(self) -> str:
        """Clear conversation history."""
        self.conversation_history = []
        self.workflow_results = []
        return "Conversation history cleared."

# Create interface
interface = MultiAgentInterface(supervisor)

# Gradio UI
with gr.Blocks(title="Day 3: Multi-Agent System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Day 3: Multi-Agent Collaboration System
    
    This system uses multiple specialized agents to process queries:
    
    - **Research Agent**: Searches and gathers information from Wikipedia and web
    - **Analysis Agent**: Analyzes and synthesizes research findings
    - **Writing Agent**: Creates professional reports and documentation
    - **Supervisor Agent**: Coordinates and delegates tasks
    
    **Available Workflows:**
    1. Sequential: Research → Analysis → Writing
    2. Supervisor: Supervisor delegates tasks dynamically
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                height=500,
                label="Conversation",
                bubble_full_width=False
            )
            
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Enter your query here...",
                    label="Query",
                    container=False,
                    scale=8
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)
            
            with gr.Row():
                workflow_dropdown = gr.Dropdown(
                    choices=["sequential", "supervisor"],
                    value="sequential",
                    label="Workflow Type",
                    scale=2
                )
                clear_btn = gr.Button("Clear", size="sm", scale=1)
                info_btn = gr.Button("System Info", size="sm", scale=1)
        
        with gr.Column(scale=1):
            info_box = gr.Markdown("### System Status\n\nReady to process queries...")
            sample_box = gr.Markdown("""
            ### Sample Queries:
            - "artificial intelligence"
            - "machine learning and deep learning"
            - "quantum computing"
            - "climate change research"
            - "Python programming"
            """)
    
    def respond_to_message(message, history, workflow_type):
        if not message or message.strip() == "":
            return history + [(message, "Please enter a query.")], ""
        
        try:
            response = interface.process_query(message, workflow_type)
            history.append((message, response))
            return history, ""
        except Exception as e:
            error_msg = f"Error: {str(e)}"
            history.append((message, error_msg))
            return history, ""
    
    def show_info():
        return interface.get_agent_info()
    
    def clear_chat():
        interface.clear_history()
        return [], "Conversation cleared."
    
    submit_btn.click(
        respond_to_message,
        inputs=[msg, chatbot, workflow_dropdown],
        outputs=[chatbot, msg]
    )
    
    msg.submit(
        respond_to_message,
        inputs=[msg, chatbot, workflow_dropdown],
        outputs=[chatbot, msg]
    )
    
    clear_btn.click(
        clear_chat,
        outputs=[chatbot, info_box]
    )
    
    info_btn.click(
        show_info,
        outputs=[info_box]
    )

print("\n" + "=" * 60)
print("Launching Multi-Agent System Interface...")
print("=" * 60)

demo.launch(share=True)

print("\n" + "=" * 60)
print("Day 3 Complete!")
print("Agents Created:")
print("  1. ResearchAgent - Searches and gathers information")
print("  2. AnalysisAgent - Analyzes and synthesizes findings")
print("  3. WritingAgent - Creates professional reports")
print("  4. SupervisorAgent - Coordinates and delegates tasks")
print("")
print("Workflows Available:")
print("  1. Sequential: Research -> Analysis -> Writing")
print("  2. Supervisor: Dynamic task delegation")
print("")
print("Interface launched successfully.")

Building Multi-Agent System Interface...

Launching Multi-Agent System Interface...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7414e831ef6e7d0bfb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Day 3 Complete!
Agents Created:
  1. ResearchAgent - Searches and gathers information
  2. AnalysisAgent - Analyzes and synthesizes findings
  3. WritingAgent - Creates professional reports
  4. SupervisorAgent - Coordinates and delegates tasks

Workflows Available:
  1. Sequential: Research -> Analysis -> Writing
  2. Supervisor: Dynamic task delegation

Interface launched successfully.


In [7]:
# CELL 7: Fix Wikipedia Search and Improve Agent Performance
# This cell fixes the Wikipedia search issues and adds better error handling

import requests
import urllib.parse
import re

print("Fixing Wikipedia Search and Improving Agents...")
print("=" * 60)

# ============================================================================
# FIXED WIKIPEDIA SEARCH FUNCTION
# ============================================================================

def fixed_wikipedia_search(query: str, max_results: int = 3) -> str:
    """
    Fixed Wikipedia search using proper API calls with better error handling.
    """
    try:
        # Use Wikipedia API with proper parameters
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": max_results,
            "srprop": "snippet|titlesnippet",
            "utf8": 1
        }
        
        headers = {
            "User-Agent": "MultiAgent-Project/1.0 (Educational Purpose)"
        }
        
        response = requests.get(url, params=params, headers=headers, timeout=10)
        
        if response.status_code != 200:
            return f"Wikipedia API error: Status {response.status_code}"
        
        data = response.json()
        results = data.get("query", {}).get("search", [])
        
        if not results:
            return f"No Wikipedia results found for: {query}"
        
        output = f"Wikipedia Results for '{query}':\n\n"
        for idx, item in enumerate(results, 1):
            title = item.get("title", "Unknown")
            snippet = item.get("snippet", "")
            # Remove HTML tags from snippet
            snippet = re.sub(r'<[^>]+>', '', snippet)
            # Get page URL
            page_url = f"https://en.wikipedia.org/wiki/{urllib.parse.quote(title.replace(' ', '_'))}"
            
            output += f"{idx}. {title}\n"
            output += f"   {snippet[:300]}\n"
            output += f"   URL: {page_url}\n\n"
        
        return output
        
    except requests.exceptions.Timeout:
        return f"Wikipedia API timeout for: {query}"
    except Exception as e:
        return f"Error searching Wikipedia: {str(e)}"

# ============================================================================
# FIXED RESEARCH AGENT
# ============================================================================

class FixedResearchAgent(BaseAgent):
    """
    Fixed Research Agent with better Wikipedia search.
    """
    
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="ResearchAgent",
            role="Information Gatherer",
            goal="Find and gather relevant information from multiple sources",
            registry=registry
        )
        self.sources_used = []
    
    def execute(self, topic: str) -> Dict[str, Any]:
        """
        Execute research on a given topic with fixed Wikipedia search.
        """
        print(f"\n[{self.name}] Researching: {topic}")
        print("-" * 40)
        
        results = {
            "topic": topic,
            "sources": [],
            "information": [],
            "timestamp": datetime.now().isoformat()
        }
        
        # Try fixed Wikipedia search first
        print("  Searching Wikipedia...")
        wiki_result = fixed_wikipedia_search(topic, max_results=3)
        if wiki_result and "No Wikipedia results" not in wiki_result and "Error" not in wiki_result:
            results["information"].append({
                "source": "Wikipedia",
                "content": wiki_result,
                "timestamp": datetime.now().isoformat()
            })
            results["sources"].append("Wikipedia")
            self.sources_used.append("Wikipedia")
            print("  Found Wikipedia information")
        else:
            print("  No Wikipedia results found")
        
        # Try Web Search as fallback
        print("  Searching Web...")
        web_result = self.execute_tool("web_search", query=topic)
        if web_result.get('success') and web_result.get('result'):
            if "No results" not in web_result['result']:
                results["information"].append({
                    "source": "Web Search",
                    "content": web_result['result'],
                    "timestamp": datetime.now().isoformat()
                })
                results["sources"].append("Web Search")
                self.sources_used.append("Web Search")
                print("  Found Web information")
            else:
                print("  No Web results found")
        
        # If no results, provide a helpful message
        if not results["information"]:
            results["information"].append({
                "source": "System",
                "content": f"No specific information found for '{topic}'.\n\nSuggestions:\n1. Try a simpler query (e.g., 'python' instead of 'python programming')\n2. Check the spelling of your query\n3. Try a different topic\n\nCommon topics that work well:\n- Artificial Intelligence\n- Machine Learning\n- Python\n- Quantum Computing",
                "timestamp": datetime.now().isoformat()
            })
        
        # Generate summary
        results["summary"] = self._generate_summary(results["information"])
        self.output = results
        
        print(f"  Research complete. Found {len(results['sources'])} sources.")
        return results
    
    def _generate_summary(self, information: List[Dict]) -> str:
        """Generate a summary of the research findings."""
        if not information:
            return "No information available."
        
        summary = f"Research found {len(information)} sources:\n"
        for idx, info in enumerate(information, 1):
            source = info.get('source', 'Unknown')
            content = info.get('content', '')
            summary += f"{idx}. {source}: {content[:150]}...\n"
        
        return summary

# ============================================================================
# FIXED ANALYSIS AGENT
# ============================================================================

class FixedAnalysisAgent(BaseAgent):
    """
    Fixed Analysis Agent with better information extraction.
    """
    
    def __init__(self, registry: ToolRegistry):
        super().__init__(
            name="AnalysisAgent",
            role="Information Analyst",
            goal="Analyze and synthesize information into structured insights",
            registry=registry
        )
    
    def execute(self, research_results: Dict[str, Any]) -> Dict[str, Any]:
        """
        Execute analysis on research results with better extraction.
        """
        print(f"\n[{self.name}] Analyzing research data...")
        print("-" * 40)
        
        topic = research_results.get('topic', 'Unknown')
        information = research_results.get('information', [])
        
        analysis = {
            "topic": topic,
            "key_findings": [],
            "insights": [],
            "recommendations": [],
            "timestamp": datetime.now().isoformat()
        }
        
        # Process each piece of information
        for info in information:
            content = info.get('content', '')
            source = info.get('source', 'Unknown')
            
            # Extract key points
            key_points = self._extract_key_points(content, source)
            if key_points:
                analysis["key_findings"].extend(key_points)
            
            # Generate insights
            insights = self._generate_insights(content, topic)
            if insights:
                analysis["insights"].extend(insights)
        
        # Remove duplicates
        analysis["key_findings"] = list(dict.fromkeys(analysis["key_findings"]))
        analysis["insights"] = list(dict.fromkeys(analysis["insights"]))
        
        # Generate recommendations
        analysis["recommendations"] = self._generate_recommendations(analysis)
        
        # Store in memory
        self.store_memory(f"Analyzed: {topic}")
        self.store_memory(f"Key findings: {len(analysis['key_findings'])}")
        self.store_memory(f"Insights: {len(analysis['insights'])}")
        
        self.output = analysis
        
        print(f"  Analysis complete. Found {len(analysis['key_findings'])} key findings.")
        print(f"  Generated {len(analysis['insights'])} insights.")
        return analysis
    
    def _extract_key_points(self, content: str, source: str) -> List[str]:
        """Extract key points from content."""
        points = []
        
        # Try to extract bullet points or numbered items
        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            if line and (line.startswith('•') or line.startswith('-') or 
                         line.startswith('1.') or line.startswith('2.') or
                         line.startswith('3.') or line.startswith('4.') or
                         line.startswith('5.') or line.startswith('*')):
                # Clean the line
                clean_line = re.sub(r'^[•\-*\d.]+', '', line).strip()
                if clean_line and len(clean_line) > 10:
                    points.append(clean_line[:200])
        
        # If no bullet points found, extract sentences
        if not points:
            sentences = re.split(r'[.!?]', content)
            for sentence in sentences[:5]:
                sentence = sentence.strip()
                if len(sentence) > 20 and len(sentence) < 200:
                    points.append(sentence)
        
        return points[:5]  # Limit to 5 key points
    
    def _generate_insights(self, content: str, topic: str) -> List[str]:
        """Generate insights from content."""
        insights = []
        content_lower = content.lower()
        topic_lower = topic.lower()
        
        # Look for connections and patterns
        if "AI" in content or "artificial intelligence" in content_lower:
            if "machine learning" in content_lower or "ML" in content_lower:
                insights.append(f"Machine learning is closely related to {topic}")
        
        if "data" in content_lower:
            insights.append(f"Data is a critical component in {topic}")
        
        if "model" in content_lower or "algorithm" in content_lower:
            insights.append(f"Algorithms and models are fundamental to {topic}")
        
        if "future" in content_lower or "development" in content_lower:
            insights.append(f"{topic} is an evolving field with future developments")
        
        # Add general insight if none found
        if not insights and len(content) > 100:
            insights.append(f"{topic} is a significant and complex subject with multiple aspects")
        
        return insights
    
    def _generate_recommendations(self, analysis: Dict) -> List[str]:
        """Generate recommendations based on analysis."""
        recommendations = []
        
        if analysis["key_findings"]:
            recommendations.append("Focus on the key findings identified")
        
        if analysis["insights"]:
            recommendations.append("Consider the insights for deeper understanding")
        
        if len(analysis["key_findings"]) < 2:
            recommendations.append("Conduct additional research on this topic")
        
        if len(analysis["insights"]) < 1:
            recommendations.append("Explore related topics for broader context")
        
        recommendations.append("Document findings for future reference")
        
        return recommendations

# ============================================================================
# UPDATE THE AGENTS
# ============================================================================

print("\nUpdating agents with fixes...")

# Create fixed agents
fixed_research_agent = FixedResearchAgent(registry)
fixed_analysis_agent = FixedAnalysisAgent(registry)
# Keep writing agent as is (it works fine)

# Update the supervisor with fixed agents
supervisor.agents['ResearchAgent'] = fixed_research_agent
supervisor.agents['AnalysisAgent'] = fixed_analysis_agent

print("  Updated ResearchAgent")
print("  Updated AnalysisAgent")
print("  WritingAgent unchanged")

print("\n" + "=" * 60)
print("Agents updated successfully.")
print("The Wikipedia search should now work better.")

Fixing Wikipedia Search and Improving Agents...

Updating agents with fixes...
  Updated ResearchAgent
  Updated AnalysisAgent
  WritingAgent unchanged

Agents updated successfully.
The Wikipedia search should now work better.


In [8]:
# CELL 8: Multi-Agent System Gradio Interface
# This cell creates a clean interface for the multi-agent system

import gradio as gr
from datetime import datetime
import json
import time

print("Building Multi-Agent System Interface...")
print("=" * 60)

class MultiAgentUI:
    """
    User interface for the multi-agent system.
    """
    
    def __init__(self, supervisor_agent):
        self.supervisor = supervisor_agent
        self.history = []
    
    def process_query(self, query: str, workflow_type: str) -> tuple:
        """
        Process a query and return formatted response.
        
        Args:
            query: User query
            workflow_type: Type of workflow to use
            
        Returns:
            Tuple of (response, status_info)
        """
        if not query or query.strip() == "":
            return "Please enter a valid query.", "Status: Error - Empty query"
        
        start_time = time.time()
        
        try:
            # Execute the appropriate workflow
            if workflow_type == "Sequential":
                result = self.supervisor.execute_sequential(query)
            else:
                result = self.supervisor.execute_supervisor_workflow(query)
            
            execution_time = time.time() - start_time
            
            # Format the response
            response = self._format_result(result, query)
            
            # Create status info
            status = f"Status: Completed in {execution_time:.2f}s | Workflow: {workflow_type}"
            
            # Store in history
            self.history.append({
                "query": query,
                "workflow": workflow_type,
                "time": execution_time,
                "timestamp": datetime.now().isoformat()
            })
            
            return response, status
            
        except Exception as e:
            error_msg = f"Error: {str(e)}"
            return error_msg, f"Status: Failed - {str(e)}"
    
    def _format_result(self, result: Dict, query: str) -> str:
        """Format the result for display."""
        lines = []
        
        lines.append("=" * 60)
        lines.append(f"QUERY: {query}")
        lines.append("=" * 60)
        lines.append("")
        
        # Get the final result
        final_result = result.get('final_result', {})
        
        if isinstance(final_result, dict):
            # Check for report
            if 'full_report' in final_result:
                lines.append(final_result['full_report'])
            elif 'report' in final_result:
                lines.append(final_result['report'])
            else:
                # Display key findings
                if 'key_findings' in final_result and final_result['key_findings']:
                    lines.append("KEY FINDINGS:")
                    lines.append("-" * 40)
                    for i, finding in enumerate(final_result['key_findings'][:5], 1):
                        lines.append(f"{i}. {finding}")
                    lines.append("")
                
                if 'insights' in final_result and final_result['insights']:
                    lines.append("INSIGHTS:")
                    lines.append("-" * 40)
                    for insight in final_result['insights'][:3]:
                        lines.append(f"  - {insight}")
                    lines.append("")
                
                if 'recommendations' in final_result and final_result['recommendations']:
                    lines.append("RECOMMENDATIONS:")
                    lines.append("-" * 40)
                    for rec in final_result['recommendations'][:3]:
                        lines.append(f"  - {rec}")
                    lines.append("")
        
        # Add workflow summary
        lines.append("")
        lines.append("-" * 60)
        lines.append("WORKFLOW SUMMARY:")
        
        if 'steps' in result:
            lines.append(f"  Steps: {len(result['steps'])}")
            for step in result['steps']:
                agent = step.get('agent', 'Unknown')
                lines.append(f"    - {agent}")
        elif 'tasks' in result:
            lines.append(f"  Tasks: {len(result['tasks'])}")
            for task in result['tasks']:
                agent = task.get('agent', 'Unknown')
                lines.append(f"    - {agent}")
        
        lines.append("=" * 60)
        
        return "\n".join(lines)
    
    def clear_history(self) -> str:
        """Clear the conversation history."""
        self.history = []
        return "History cleared."
    
    def get_history_summary(self) -> str:
        """Get a summary of conversation history."""
        if not self.history:
            return "No queries processed yet."
        
        lines = []
        lines.append("=" * 60)
        lines.append("CONVERSATION HISTORY")
        lines.append("=" * 60)
        
        for i, entry in enumerate(self.history[-5:], 1):
            lines.append(f"{i}. Query: {entry['query'][:50]}...")
            lines.append(f"   Workflow: {entry['workflow']}")
            lines.append(f"   Time: {entry['time']:.2f}s")
            lines.append("")
        
        lines.append(f"Total queries: {len(self.history)}")
        lines.append("=" * 60)
        
        return "\n".join(lines)

# Create the UI instance
ui = MultiAgentUI(supervisor)

# Gradio interface
with gr.Blocks(title="Multi-Agent System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Multi-Agent Collaboration System
    
    **How it works:**
    1. **Research Agent** - Gathers information from Wikipedia and web
    2. **Analysis Agent** - Analyzes and synthesizes findings
    3. **Writing Agent** - Creates professional reports
    4. **Supervisor Agent** - Coordinates all agents
    
    **Workflows:**
    - **Sequential**: Research → Analysis → Writing (one after another)
    - **Supervisor**: Supervisor delegates tasks dynamically
    """)
    
    with gr.Row():
        with gr.Column(scale=2):
            # Chatbot
            chatbot = gr.Chatbot(
                height=450,
                label="Conversation",
                bubble_full_width=False
            )
            
            # Input area
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Enter your query (e.g., 'artificial intelligence', 'python programming')...",
                    label="Query",
                    container=False,
                    scale=7
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)
            
            # Controls
            with gr.Row():
                workflow_dropdown = gr.Dropdown(
                    choices=["Sequential", "Supervisor"],
                    value="Sequential",
                    label="Workflow Type",
                    scale=2
                )
                clear_btn = gr.Button("Clear Chat", size="sm", scale=1)
                history_btn = gr.Button("Show History", size="sm", scale=1)
        
        with gr.Column(scale=1):
            # Status and info
            status_box = gr.Markdown("**Status:** Ready")
            info_box = gr.Markdown("""
            ### Quick Tips
            
            **Good queries to try:**
            - "artificial intelligence"
            - "machine learning"
            - "python programming"
            - "quantum computing"
            - "climate change"
            
            **For best results:**
            - Use specific topics
            - Avoid very long queries
            - Be clear and concise
            """)
    
    def respond(message, history, workflow_type):
        if not message or message.strip() == "":
            return history, "Please enter a valid query."
        
        response, status = ui.process_query(message, workflow_type)
        history.append((message, response))
        return history, status
    
    def show_history():
        return ui.get_history_summary()
    
    def clear_chat():
        ui.clear_history()
        return [], "History cleared."
    
    # Connect events
    submit_btn.click(
        respond,
        inputs=[msg, chatbot, workflow_dropdown],
        outputs=[chatbot, status_box]
    )
    
    msg.submit(
        respond,
        inputs=[msg, chatbot, workflow_dropdown],
        outputs=[chatbot, status_box]
    )
    
    clear_btn.click(
        clear_chat,
        outputs=[chatbot, status_box]
    )
    
    history_btn.click(
        show_history,
        outputs=[status_box]
    )

print("\n" + "=" * 60)
print("Launching Multi-Agent System Interface...")
print("=" * 60)

demo.launch(share=True)

print("\n" + "=" * 60)
print("DAY 3 COMPLETE!")
print("=" * 60)
print("Agents Created:")
print("  - ResearchAgent: Searches Wikipedia and web")
print("  - AnalysisAgent: Analyzes and synthesizes findings")
print("  - WritingAgent: Creates professional reports")
print("  - SupervisorAgent: Coordinates and delegates tasks")
print("")
print("Workflows:")
print("  - Sequential: Research -> Analysis -> Writing")
print("  - Supervisor: Dynamic task delegation")
print("")
print("Try queries like:")
print("  - 'artificial intelligence'")
print("  - 'python programming'")
print("  - 'quantum computing'")
print("=" * 60)

Building Multi-Agent System Interface...

Launching Multi-Agent System Interface...
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://481158414cbf0937b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



DAY 3 COMPLETE!
Agents Created:
  - ResearchAgent: Searches Wikipedia and web
  - AnalysisAgent: Analyzes and synthesizes findings
  - WritingAgent: Creates professional reports
  - SupervisorAgent: Coordinates and delegates tasks

Workflows:
  - Sequential: Research -> Analysis -> Writing
  - Supervisor: Dynamic task delegation

Try queries like:
  - 'artificial intelligence'
  - 'python programming'
  - 'quantum computing'

SEQUENTIAL WORKFLOW
Query: gpt vs llm

[Step 1] Research Agent
----------------------------------------

[Supervisor] Delegating to: ResearchAgent

[ResearchAgent] Researching: gpt vs llm
----------------------------------------
  Searching Wikipedia...
  Found Wikipedia information
  Searching Web...
  Found Web information
  Research complete. Found 2 sources.

[Step 2] Analysis Agent
----------------------------------------

[Supervisor] Delegating to: AnalysisAgent

[AnalysisAgent] Analyzing research data...
----------------------------------------
  Analysis